In [16]:
from pathlib import Path
from IPython.display import Markdown, display

def generate_tree(
    root=".",
    show_files=False,
    exclude_dirs=None,
    exclude_files=None,
    collapse_dirs=None,
    include_dirs=None,
    include_files=None,
):
    """
    Generate a markdown tree representation of a directory.

    Parameters
    ----------
    root : str
        root folder to scan
    show_files : bool
        if False -> show folders only
    exclude_dirs : list[str]
        directories to ignore completely
    exclude_files : list[str]
        files to ignore
    collapse_dirs : list[str]
        directories that appear but are not expanded
    include_dirs : list[str]
        directories that should always appear even if excluded
    include_files : list[str]
        files that should always appear even if excluded
    """

    root = Path(root)

    exclude_dirs = set(exclude_dirs or [])
    exclude_files = set(exclude_files or [])
    collapse_dirs = set(collapse_dirs or [])
    include_dirs = set(include_dirs or [])
    include_files = set(include_files or [])

    lines = []

    def walk(folder, prefix=""):

        entries = sorted(folder.iterdir(), key=lambda x: (x.is_file(), x.name.lower()))

        for i, entry in enumerate(entries):

            connector = "└── " if i == len(entries)-1 else "├── "

            if entry.is_dir():

                if entry.name in exclude_dirs and entry.name not in include_dirs:
                    continue

                lines.append(prefix + connector + entry.name)

                if entry.name not in collapse_dirs:
                    extension = "    " if i == len(entries)-1 else "│   "
                    walk(entry, prefix + extension)

            else:

                if not show_files and entry.name not in include_files:
                    continue

                if entry.name in exclude_files and entry.name not in include_files:
                    continue

                lines.append(prefix + connector + entry.name)

    lines.append(root.name)
    walk(root)

    return "```\n" + "\n".join(lines) + "\n```"

In [20]:
def build_project_tree(
    show_files=False,
    exclude_dirs=[".venv", "__pycache__", ".git"],
    exclude_files=None,
    collapse_dirs=None,
    include_dirs=None,
    include_files=None,
):
    
    tree_md = generate_tree(
        root=".",
        show_files=show_files,
        exclude_dirs=exclude_dirs,
        exclude_files=exclude_files,
        collapse_dirs=collapse_dirs,
        include_dirs=include_dirs,
        include_files=include_files,
    )

    display(Markdown(tree_md))
    #return tree_md

In [21]:
build_project_tree()

```

├── .vscode
├── daily-exercises
│   ├── Raw
│   ├── speed-drills
│   │   ├── notebooks
├── notes
│   └── Raw
├── picked_up_from_somewhere
├── question-banks
├── Rough
```

Example usages inside the notebook.

1️⃣ Folder structure only

```build_project_tree(show_files=False)```

2️⃣ Full tree

`build_project_tree(show_files=True)`

3️⃣ Collapse large folders

`build_project_tree(
    show_files=True,
    collapse_dirs=[".venv", "__pycache__", ".git"]
)`

4️⃣ Hide certain files

`build_project_tree(
    show_files=True,
    exclude_files=[".DS_Store"]
)`

Hide .venv but show one file inside it

`build_project_tree(
    show_files=True,
    exclude_dirs=[".venv"],
    include_files=["pyvenv.cfg"]
)`

Collapse .venv but still display it

`build_project_tree(
    show_files=True,
    collapse_dirs=[".venv"]
)`

Hide most files but force include specific ones

`build_project_tree(
    show_files=False,
    include_files=["README.md", "pyproject.toml"]
)`